# Core species analysis - primary notebook

In [3]:
import polars as pl

In [4]:
df = (
    pl.scan_parquet('../outputs.cds3/pq/mag+gtdb.cds3.x.3216.manysearch.parquet')
    .filter(pl.col('intersect_hashes') >= 20)
).collect()

In [5]:
df

query_name,query_md5,match_name,containment,intersect_hashes,ksize,scaled,moltype,match_md5,jaccard,max_containment,average_abund,median_abund,std_abund,query_containment_ani,match_containment_ani,average_containment_ani,max_containment_ani,n_weighted_found,total_weighted_hashes
str,str,str,f64,i64,i64,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
"""GCA_963606895 s__Methanocatell…","""98787da9033ca08f4a8e0e553cd3be…","""SRR11183330""",0.18467,465,21,1000,"""DNA""","""91703afddeec80707490192525488c…",0.001806,0.18467,10.012903,10.0,5.956098,0.922713,0.740505,0.831609,0.922713,4656,1066419
"""GCF_902163005 s__Enterococcus …","""ca79ba3d2b584c0ffdc81940fc1984…","""SRR11183330""",0.003933,63,21,1000,"""DNA""","""91703afddeec80707490192525488c…",0.000232,0.003933,1.634921,1.0,1.87973,0.768183,0.67327,0.720727,0.768183,103,1066419
"""GCF_001256715 s__Escherichia c…","""2e7f136cae1634a43c52b78cb970e0…","""SRR11183330""",0.017325,2041,21,1000,"""DNA""","""91703afddeec80707490192525488c…",0.005498,0.017325,10.982852,2.0,34.046445,0.824381,0.794544,0.809462,0.824381,22416,1066419
"""GCA_900513975 s__Klebsiella pn…","""caef9e116e27e609502c15f8f56877…","""SRR11183330""",0.000643,27,21,1000,"""DNA""","""91703afddeec80707490192525488c…",0.000091,0.000643,3.888889,1.0,9.488134,0.704729,0.646646,0.675688,0.704729,105,1066419
"""GCA_034118145 s__Butyricimonas…","""c670dcee3e5cbe426271f4a3a7b589…","""SRR11183330""",0.101147,291,21,1000,"""DNA""","""91703afddeec80707490192525488c…",0.001128,0.101147,2.168385,2.0,1.236264,0.896637,0.72416,0.810399,0.896637,631,1066419
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""GCA_041675925 s__Paracholeplas…","""03f6ba47de50e2dd2a1ff2d60c24a5…","""SRR14369146""",0.025259,56,21,1000,"""DNA""","""2f319be4771fc5a4bfba076b7fb367…",0.000028,0.025259,3.5,2.5,3.570714,0.839315,0.607259,0.723287,0.839315,196,7255242
"""GCA_017508245 s__NHYM01 sp0175…","""61144402e05944f3e97c3969a96231…","""SRR14369146""",0.134986,49,21,1000,"""DNA""","""2f319be4771fc5a4bfba076b7fb367…",0.000025,0.134986,1.367347,1.0,0.522704,0.909045,0.60341,0.756227,0.909045,67,7255242
"""GCA_934295335 s__Eubacterium_F…","""6fa54e46f6187a0282c6cddc1d3048…","""SRR14369146""",0.018105,64,21,1000,"""DNA""","""2f319be4771fc5a4bfba076b7fb367…",0.000032,0.018105,1.59375,1.0,0.878898,0.82611,0.611133,0.718621,0.82611,102,7255242


In [6]:
n_acc = df['match_name'].n_unique()
by_species = df.group_by('query_name').agg(
    freq=pl.len() / n_acc
)

In [7]:
with pl.Config(tbl_rows=-1):
    print(by_species.sort('freq', descending=True).filter(pl.col('freq') >= 0.95))

shape: (17, 2)
┌─────────────────────────────────┬──────────┐
│ query_name                      ┆ freq     │
│ ---                             ┆ ---      │
│ str                             ┆ f64      │
╞═════════════════════════════════╪══════════╡
│ GCF_001256715 s__Escherichia c… ┆ 0.992537 │
│ GCA_945834365 s__Sodaliphilus … ┆ 0.990983 │
│ GCF_028308525 s__Lactobacillus… ┆ 0.990361 │
│ GCA_946007015 s__UBA2868 sp004… ┆ 0.979167 │
│ GCA_927798655 s__Cryptobactero… ┆ 0.978545 │
│ GCA_034171835 s__Mogibacterium… ┆ 0.977301 │
│ GCA_945872445 s__JAFBIX01 sp02… ┆ 0.974502 │
│ GCA_946408405 s__Fimisoma sp00… ┆ 0.973259 │
│ GCA_945501735 s__Prevotella sp… ┆ 0.968595 │
│ GCA_945876695 s__Floccifex por… ┆ 0.968284 │
│ GCA_004561115 s__Cryptobactero… ┆ 0.968284 │
│ GCA_945877245 s__Bariatricus s… ┆ 0.966418 │
│ GCA_022781365 s__Colivicinus s… ┆ 0.958333 │
│ GCA_945932635 s__Cryptobactero… ┆ 0.956468 │
│ GCA_945938975 s__Ornithospiroc… ┆ 0.955224 │
│ GCA_934726065 s__Cryptobactero… ┆ 0.952425 

In [8]:
OLD_NAMES = [ x.strip() for x in open('../../2025-workflow-core99/inputs.cds/names.list') ]
OLD_NAMES_PLUS = [ x.strip() for x in open('../../2025-workflow-core99/inputs.cds/names-plus.list') ]

In [9]:
OLD_NAMES

['s__Bariatricus sp004560705',
 's__Colivicinus sp002299675',
 's__Cryptobacteroides sp000432655',
 's__Cryptobacteroides sp000434935',
 's__Cryptobacteroides sp034089285',
 's__Cryptobacteroides sp900546925',
 's__Fimisoma sp002320005',
 's__Floccifex porci',
 's__JAFBIX01 sp021531895',
 's__Lactobacillus amylovorus',
 's__Mogibacterium_A kristiansenii',
 's__Ornithospirochaeta sp022785155',
 's__Prevotella sp000434975',
 's__Prevotella sp002251295',
 's__Sodaliphilus sp004557565',
 's__UBA2868 sp004552595']

In [10]:
xx_df = (
    by_species
    .with_columns(species_name=pl.col('query_name').str.split(' ').list.slice(1, 2).list.join( " "))
).filter(~(pl.col('species_name').is_in(OLD_NAMES)))
xx_df.filter(pl.col('freq') >= 0.95)

query_name,freq,species_name
str,f64,str
"""GCF_001256715 s__Escherichia c…",0.992537,"""s__Escherichia coli"""


In [11]:
len(OLD_NAMES)

16

In [12]:
xx_df = (
    by_species
    .with_columns(species_name=pl.col('query_name').str.split(' ').list.slice(1, 2).list.join( " "))
).filter((pl.col('species_name').is_in(OLD_NAMES_PLUS)))
with pl.Config(tbl_rows=-1):
    print(xx_df.sort('freq', descending=True))

shape: (27, 3)
┌─────────────────────────────────┬──────────┬─────────────────────────────────┐
│ query_name                      ┆ freq     ┆ species_name                    │
│ ---                             ┆ ---      ┆ ---                             │
│ str                             ┆ f64      ┆ str                             │
╞═════════════════════════════════╪══════════╪═════════════════════════════════╡
│ GCA_945834365 s__Sodaliphilus … ┆ 0.990983 ┆ s__Sodaliphilus sp004557565     │
│ GCF_028308525 s__Lactobacillus… ┆ 0.990361 ┆ s__Lactobacillus amylovorus     │
│ GCA_946007015 s__UBA2868 sp004… ┆ 0.979167 ┆ s__UBA2868 sp004552595          │
│ GCA_927798655 s__Cryptobactero… ┆ 0.978545 ┆ s__Cryptobacteroides sp9005469… │
│ GCA_034171835 s__Mogibacterium… ┆ 0.977301 ┆ s__Mogibacterium_A kristiansen… │
│ GCA_945872445 s__JAFBIX01 sp02… ┆ 0.974502 ┆ s__JAFBIX01 sp021531895         │
│ GCA_946408405 s__Fimisoma sp00… ┆ 0.973259 ┆ s__Fimisoma sp002320005         │
│ GCA_9455017

In [13]:
len(OLD_NAMES_PLUS)

27

## Ask questions about core species in metagenomes by metagenome

In [14]:
by_acc = (
    df
    .with_columns(
        species_name=pl.col('query_name').str.split(' ').list.slice(1, 2).list.join( " ")
    )
    .filter(pl.col('species_name').is_in(OLD_NAMES))
    .group_by('match_name').agg(
        n_core=pl.len()
    )
).sort('n_core')


In [15]:
by_acc


match_name,n_core
str,u32
"""SRR12795729""",1
"""SRR12795774""",1
"""SRR12795782""",1
"""SRR12795740""",1
"""SRR12795783""",1
…,…
"""SRR11183573""",16
"""SRR11126323""",16
"""SRR17241519""",16


In [16]:
low_core_acc = by_acc.filter(pl.col('n_core') < 10)

In [17]:
metadata_df = (
    pl.scan_parquet("/group/ctbrowngrp5/sra-metagenomes/20241128-metadata.parquet")
    .filter(pl.col("acc") != "NP")
    .filter(pl.col("assay_type") == "WGS")
#    .select(["acc", "organism", "bioproject", "mbases", "host", "project_name"]) 
    .collect()
)

In [18]:
low_core_acc = low_core_acc.rename({'match_name': 'acc'})

In [19]:
low_core_acc2 = low_core_acc.join(metadata_df, on='acc', how='left')
low_core_acc2

acc,n_core,sample_name,sample_name_sam,assay_type,avgspotlen,bioproject,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search
str,u32,str,list[str],str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR12795729""",1,"""pig_colon_412""",[],"""WGS""",302,"""PRJNA668104""","""SAMN16396807""","[""Metagenome or environmental""]","""UNIVERSITY OF COPENHAGEN""",2018-09-07,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX9264850""","""Denmark""","""Europe""","[""Denmark:Copenhagen""]",null,"""Illumina NovaSeq 6000""","""17119-05-13-243200076""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,7340,2352,"""pig gut metagenome""","""ILLUMINA""",2021-10-01 00:00:00 UTC,"""SRS7494397""","""SRP286761""",null,null,null,null,"""[""2018-09-07""]""",null,null,null,null,null,null,null,null,"""[""Crossbred piglets (Landrace …",null,null,null,null,null,null,null,null,null,"""""55.68 N 12.54 E""""",null,null,null,null,"""7340447032""","""2467034839""","""""2020-10-08T12:26:00.000Z""""",null,"""""16396807"""""
"""SRR12795774""",1,"""pig_colon_406""",[],"""WGS""",302,"""PRJNA668104""","""SAMN16396805""","[""Metagenome or environmental""]","""UNIVERSITY OF COPENHAGEN""",2018-09-07,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX9264805""","""Denmark""","""Europe""","[""Denmark:Copenhagen""]",null,"""Illumina NovaSeq 6000""","""17119-05-07-243200099""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,6253,1981,"""pig gut metagenome""","""ILLUMINA""",2021-10-01 00:00:00 UTC,"""SRS7494382""","""SRP286761""",null,null,null,null,"""[""2018-09-07""]""",null,null,null,null,null,null,null,null,"""[""Crossbred piglets (Landrace …",null,null,null,null,null,null,null,null,null,"""""55.68 N 12.54 E""""",null,null,null,null,"""6253850126""","""2077501527""","""""2020-10-08T12:22:00.000Z""""",null,"""""16396805"""""
"""SRR12795782""",1,"""pig_colon_424""",[],"""WGS""",302,"""PRJNA668104""","""SAMN16396824""","[""Metagenome or environmental""]","""UNIVERSITY OF COPENHAGEN""",2018-09-07,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX9264797""","""Denmark""","""Europe""","[""Denmark:Copenhagen""]",null,"""Illumina NovaSeq 6000""","""17119-05-25-243200085""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,8907,2766,"""pig gut metagenome""","""ILLUMINA""",2021-10-01 00:00:00 UTC,"""SRS7494374""","""SRP286761""",null,null,null,null,"""[""2018-09-07""]""",null,null,null,null,null,null,null,null,"""[""Crossbred piglets (Landrace …",null,null,null,null,null,null,null,null,null,"""""55.68 N 12.54 E""""",null,null,null,null,"""8907858440""","""2900407656""","""""2020-10-08T12:24:00.000Z""""",null,"""""16396824"""""
"""SRR12795740""",1,"""pig_colon_409""",[],"""WGS""",302,

In [20]:
low_core_acc2.group_by('bioproject').agg(
    pl.len()
).sort('len')

bioproject,len
str,u32
"""PRJNA741980""",1
"""PRJNA629856""",1
"""PRJNA471402""",1
"""PRJNA408025""",3
"""PRJEB31650""",5
"""PRJNA807368""",7
"""PRJNA373834""",8
"""PRJNA526405""",14
"""PRJNA668104""",17


## Explore saturation

In [21]:
sat_df = pl.read_parquet('../../2025-ccbaumler-wort-gathering/wort-sra-signature-saturation.parquet')

In [22]:
sat_df

filepath,signature,novel_count,total_count,sequence_saturation
str,str,i64,i64,f64
"""/group/ctbrowngrp/irber/data/w…","""SRR33867156""",23507,397539,0.940869
"""/group/ctbrowngrp/irber/data/w…","""ERR4022268""",27896,308529,0.909584
"""/group/ctbrowngrp/irber/data/w…","""ERR3771976""",223,3174,0.929742
"""/group/ctbrowngrp/irber/data/w…","""ERR11996469""",20,40,0.5
"""/group/ctbrowngrp/irber/data/w…","""ERR10755277""",3185,26976,0.881932
…,…,…,…,…
"""/group/ctbrowngrp/irber/data/w…","""SRR9934015""",57128,286525,0.800618
"""/group/ctbrowngrp/irber/data/w…","""SRR7588346""",67307,409191,0.835512
"""/group/ctbrowngrp/irber/data/w…","""SRR10432619""",88275,942001,0.90629


In [23]:
sat_df2 = df.join(sat_df, left_on='match_name', right_on='signature',
                  how='left')

In [24]:
zz = []
for min_sat in (0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    xx_df = sat_df2.filter(pl.col('sequence_saturation') >= min_sat)
    n_acc = xx_df['match_name'].n_unique()
    xx2_df= df.group_by('query_name').agg(
        freq=pl.len() / n_acc
    ).filter(pl.col('freq') >= 0.95)
    zz.append(dict(min=min_sat, num_metag=n_acc, num_core95=len(xx2_df)))
    #print(min_sat, n_acc, len(xx2_df))

zz_df = pl.DataFrame(zz)


In [25]:
zz_df

min,num_metag,num_core95
f64,i64,i64
0.25,3216,17
0.3,3215,17
0.4,3206,18
0.5,3065,38
0.6,2251,269
0.7,1244,737
0.8,366,1724
0.9,37,3116
